<a href="https://colab.research.google.com/github/mukulhayaran/pyTorch-Learning/blob/main/PyTorch_TransferLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import AlexNet, AlexNet_Weights
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from tqdm import tqdm
import numpy as np
from PIL import Image, UnidentifiedImageError

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Download the dataset directly
!wget -O cats_vs_dogs.zip "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"

# Unzip it silently into a folder named 'dataset_folder'
!unzip -q cats_vs_dogs.zip -d dataset_folder

--2026-03-24 16:25:50--  https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip
Resolving download.microsoft.com (download.microsoft.com)... 23.62.134.55, 2600:1406:5400:2ac::317f, 2600:1406:5400:2ae::317f
Connecting to download.microsoft.com (download.microsoft.com)|23.62.134.55|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 824887076 (787M) [application/octet-stream]
Saving to: ‘cats_vs_dogs.zip’

cats_vs_dogs.zip    100%[===================>] 786.67M   163MB/s    in 6.9s    

2026-03-24 16:25:57 (113 MB/s) - ‘cats_vs_dogs.zip’ saved [824887076/824887076]



In [25]:
PATH_TO_DATA = "dataset_folder/PetImages/"

normalizer= transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])

train_transforms = transforms.Compose(
    [
     transforms.Resize((224,224)),
     transforms.RandomHorizontalFlip(p=0.5),
     transforms.ToTensor(),

     normalizer

    ]
)

dataset = ImageFolder(PATH_TO_DATA,transform=train_transforms)

# Filter out corrupted images
print("Filtering out corrupted images...")
valid_indices = []
for i in tqdm(range(len(dataset))):
    try:
        # Attempt to load the image to catch UnidentifiedImageError
        _ = dataset[i]
        valid_indices.append(i)
    except UnidentifiedImageError:
        # Skip corrupted images
        pass

# Create a new dataset subset with only valid images
filtered_dataset = torch.utils.data.Subset(dataset, valid_indices)

train_samples,test_samples=int(0.9*len(filtered_dataset)), len(filtered_dataset)-int( 0.9*len(filtered_dataset))
train_dataset, val_dataset= torch.utils.data.random_split(filtered_dataset, lengths=[train_samples,test_samples])

# Remove the sample loop to avoid error from filtered dataset
# for sample in dataset:
#   print(sample)
#   break

Filtering out corrupted images...


100%|██████████| 25000/25000 [01:26<00:00, 287.75it/s]


In [ ]:
PATH_TO_DATA = "dataset_folder/"

normalizer= transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])

train_transforms = transforms.Compose(
    [
     transforms.Resize((224,224)),
     transforms.RandomHorizontalFlip(p=0.5),
     transforms.ToTensor(),

     normalizer

    ]
)

dataset = ImageFolder(PATH_TO_DATA,transform=train_transforms)

# Filter out corrupted images
print("Filtering out corrupted images...")
valid_indices = []
for i in tqdm(range(len(dataset))):
    try:
        # Attempt to load the image to catch UnidentifiedImageError
        _ = dataset[i]
        valid_indices.append(i)
    except UnidentifiedImageError:
        # Skip corrupted images
        pass

# Create a new dataset subset with only valid images
filtered_dataset = torch.utils.data.Subset(dataset, valid_indices)

train_samples,test_samples=int(0.9*len(filtered_dataset)), len(filtered_dataset)-int( 0.9*len(filtered_dataset))
train_dataset, val_dataset= torch.utils.data.random_split(filtered_dataset, lengths=[train_samples,test_samples])

# Remove the sample loop to avoid error from filtered dataset
# for sample in dataset:
#   print(sample)
#   break

Filtering out corrupted images...


 42%|████▏     | 10604/25000 [00:36<00:45, 315.97it/s]

In [4]:
model = AlexNet()
model

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

In [5]:
model.classifier

Sequential(
  (0): Dropout(p=0.5, inplace=False)
  (1): Linear(in_features=9216, out_features=4096, bias=True)
  (2): ReLU(inplace=True)
  (3): Dropout(p=0.5, inplace=False)
  (4): Linear(in_features=4096, out_features=4096, bias=True)
  (5): ReLU(inplace=True)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)

In [6]:
model.classifier[6]= nn.Linear(4096,2)
model

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

In [7]:
rand_data=torch.randn(16,3,224,224)

model(rand_data).shape

torch.Size([16, 2])

In [8]:
num_params=0
for name, param in model.named_parameters():
  num_params+=param.numel()


num_params



57012034

In [26]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("training on device", DEVICE)

model= AlexNet()
model.classifier[6]=torch.nn.Linear(4096,2)
model = model.to(DEVICE)

epochs = 2
optimizer= optim.Adam(model.parameters(), lr=0.0001)
loss= nn.CrossEntropyLoss()
batch_size=128
train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True, num_workers=4)
val_loader=DataLoader(val_dataset,batch_size=batch_size, shuffle=True, num_workers=4)



def train(model,device, epochs, optimizer, loss_fn,batch_size,trainloader,valloader):

  log_training= {
      "epoch":[],
      "training_loss":[],
      "training_acc":[],
      "validation_loss":[],
      "validation_acc":[]
  }

  for epoch in range(epochs):

    print(f"starting epoch {epoch+1}")

    training_losses,training_accuracies=[],[]
    validation_losses,validation_accuracies=[],[]

    model.train()
    for image,label in tqdm(trainloader):
      image,label=image.to(device), label.to(device)

      out=model(image)

      loss=loss_fn(out,label)
      training_losses.append(loss.item())

      #compute accuracy #
      predictions=torch.argmax(out,axis=-1)
      accuracy=(predictions==label).sum()/len(predictions)
      training_accuracies.append(accuracy.cpu().item())


      loss.backward()
      optimizer.step()
      optimizer.zero_grad()


    model.eval() #dropout is turned off during eval
    for image,label in tqdm(valloader):
        image,label=image.to(device), label.to(device)


        with torch.inference_mode():
          out=model(image)

        loss=loss_fn(out,label)
        validation_losses.append(loss.item())

        #compute accuracy #
        predictions=torch.argmax(out,axis=-1)
        accuracy=(predictions==label).sum()/len(predictions)
        validation_accuracies.append(accuracy.cpu().item())


    training_loss_mean=np.mean(training_losses)
    training_acc_mean=np.mean(training_accuracies)
    validation_loss_mean=np.mean(validation_losses)
    validation_acc_mean=np.mean(validation_accuracies)

    log_training["epoch"].append(epoch)
    log_training["training_loss"].append(training_loss_mean)
    log_training["training_acc"].append(training_acc_mean)
    log_training["validation_loss"].append(validation_loss_mean)
    log_training["validation_acc"].append(validation_acc_mean)

    print("training_loss:", training_loss_mean)
    print("training_acc:", training_acc_mean)
    print("validation_loss:", validation_loss_mean)
    print("validation_acc:", validation_acc_mean)


  return log_training,model


random_init_log, model=train(model, DEVICE, epochs,optimizer,loss,batch_size,train_loader,val_loader)

training on device cuda
starting epoch 1


100%|██████████| 20/20 [00:08<00:00,  2.43it/s]


training_loss: 0.591523792933334
training_acc: 0.6739015030589971
validation_loss: 0.5063528418540955
validation_acc: 0.7606847435235977
starting epoch 2


100%|██████████| 20/20 [00:09<00:00,  2.09it/s]

training_loss: 0.45619486255401914
training_acc: 0.782481121068651
validation_loss: 0.4232263550162315
validation_acc: 0.8091911762952805


In [24]:
# Check class distribution in validation dataset
val_labels = []
for _, label in val_dataset:
    val_labels.append(label)

unique_labels, counts = np.unique(val_labels, return_counts=True)
label_distribution = dict(zip(unique_labels, counts))

print("Validation dataset class distribution:", label_distribution)

# Get class names from the original dataset
class_names = dataset.classes
print("Class names:", class_names)

Validation dataset class distribution: {np.int64(0): np.int64(2500)}
Class names: ['PetImages']


In [19]:
random_init_log


{'epoch': [1],
 'training_loss': [np.float64(0.0)],
 'training_acc': [np.float64(1.0)],
 'validation_loss': [np.float64(1.0)],
 'validation_acc': [np.float64(1.0)]}

In [28]:
#loading pretrained weights

model= torch.hub.load("pytorch/vision:v0.10.0","alexnet",pretrained=True)

model.classifier[6]=nn.Linear(4096,2)
model=model.to(DEVICE)

epochs =2
optimizer= optim.Adam(model.parameters(), lr=0.0001)
loss_fn=nn.CrossEntropyLoss()
batch_size=128
train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True, num_workers=4)
val_loader=DataLoader(val_dataset,batch_size=batch_size, shuffle=True, num_workers=4)

pre_init_log,model=train(model,DEVICE,epochs,optimizer,loss_fn,batch_size,train_loader,val_loader)



Using cache found in /root/.cache/torch/hub/pytorch_vision_v0.10.0


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 152MB/s]


starting epoch 1


100%|██████████| 20/20 [00:10<00:00,  1.85it/s]


training_loss: 0.12343158972957595
training_acc: 0.9505984386937185
validation_loss: 0.10505030173808336
validation_acc: 0.9606847435235977
starting epoch 2


100%|██████████| 20/20 [00:10<00:00,  1.91it/s]


training_loss: 0.06930703659203243
training_acc: 0.973431702025912
validation_loss: 0.07478563990443945
validation_acc: 0.9703584551811218


In [29]:
#freezing weights:

for name,param in model.named_parameters():
  print(name)

features.0.weight
features.0.bias
features.3.weight
features.3.bias
features.6.weight
features.6.bias
features.8.weight
features.8.bias
features.10.weight
features.10.bias
classifier.1.weight
classifier.1.bias
classifier.4.weight
classifier.4.bias
classifier.6.weight
classifier.6.bias


In [31]:

model= torch.hub.load("pytorch/vision:v0.10.0","alexnet",pretrained=True)

model.classifier[6]=nn.Linear(4096,2)
model=model.to(DEVICE)

for name,param in model.named_parameters():
  if "classifier.6" not in name:
    param.requires_grad_(False)

epochs =2
optimizer= optim.Adam(model.parameters(), lr=0.0001)
loss_fn=nn.CrossEntropyLoss()
batch_size=128
train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True, num_workers=4)
val_loader=DataLoader(val_dataset,batch_size=batch_size, shuffle=True, num_workers=4)

pre_init_log,model=train(model,DEVICE,epochs,optimizer,loss_fn,batch_size,train_loader,val_loader)



Using cache found in /root/.cache/torch/hub/pytorch_vision_v0.10.0


starting epoch 1


100%|██████████| 20/20 [00:10<00:00,  1.98it/s]


training_loss: 0.22384787841954015
training_acc: 0.9065715951675718
validation_loss: 0.12853120304644108
validation_acc: 0.9537454038858414
starting epoch 2


100%|██████████| 20/20 [00:08<00:00,  2.42it/s]

training_loss: 0.12958097874864258
training_acc: 0.9499090473082933
validation_loss: 0.10865223221480846
validation_acc: 0.9619025737047195
